In [5]:
!pip install -q diffusers transformers accelerate safetensors gradio controlnet-aux opencv-python

import torch
import gradio as gr
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline
from controlnet_aux import MidasDetector

device = "cuda" if torch.cuda.is_available() else "cpu"

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
).to(device)

depth_estimator = MidasDetector.from_pretrained("lllyasviel/Annotators")

def modify_image(input_image, prompt, strength, guidance_scale, steps):
    if input_image is None:
        return None

    strength = float(strength)
    guidance_scale = float(guidance_scale)
    steps = int(steps)

    # resize to multiple of 8
    w, h = input_image.size
    w -= w % 8
    h -= h % 8
    input_image = input_image.resize((w, h))

    depth_map = depth_estimator(input_image).resize((w, h))

    result = pipe(
        prompt=prompt,
        negative_prompt="snow, white background, fog, faceless, mannequin head, distorted body, blurry, low quality",
        image=input_image,
        control_image=depth_map,
        strength=strength,
        guidance_scale=guidance_scale,
        num_inference_steps=steps,
    )

    return result.images[0]

# Gradio UI
demo = gr.Interface(
    fn=modify_image,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Textbox(label="Prompt"),
        gr.Slider(0.1, 1.0, value=0.55, label="Strength"),
        gr.Slider(1, 15, value=11, label="Guidance Scale"),
        gr.Slider(10, 50, value=40, label="Steps"),
    ],
    outputs=gr.Image(label="Output"),
    title="Text-Triggered Image Editing (ControlNet)",
)

demo.launch(debug=False)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 7.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_5m_224 in registry with controlnet_aux.segmen

config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

safety_checker/model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

dpt_hybrid-midas-501f0c75.pt:   0%|          | 0.00/493M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cadc4d79a8c5a71f6a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


 ## Provided image and generated image are in the folder under Aman

